In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import trange, tqdm
import copy

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder, LabelBinarizer
from sklearn.model_selection import train_test_split

from scipy.fft import fft

import torch

# **Load Dataset**

In [102]:
df = pd.read_csv("F:/Project-Motor Imagery Classification/BCICIV_2a_gdf/CSVs/BCICIV_2a_all_patients.csv/BCICIV_2a_all_patients.csv")
# Drop NA just in case (though this dataset has none)
df = df.dropna()
# EEG columns (starting from column 4 onwards)
eeg_columns = df.columns[4:]

print("Channels used:", eeg_columns)

Channels used: Index(['EEG-Fz', 'EEG-0', 'EEG-1', 'EEG-2', 'EEG-3', 'EEG-4', 'EEG-5',
       'EEG-C3', 'EEG-6', 'EEG-Cz', 'EEG-7', 'EEG-C4', 'EEG-8', 'EEG-9',
       'EEG-10', 'EEG-11', 'EEG-12', 'EEG-13', 'EEG-14', 'EEG-Pz', 'EEG-15',
       'EEG-16'],
      dtype='object')


In [103]:
df["patient"].unique()

array([1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=int64)

In [104]:
#ONLY for SUBJECT-WISE data
sub=9
df = df[df["patient"]!=sub]
df = df.reset_index(drop=True)
df = df.dropna()

In [105]:
df.head(10)

,patient,time,label,epoch,EEG-Fz,EEG-0,EEG-1,EEG-2,EEG-3,EEG-4,...,EEG-8,EEG-9,EEG-10,EEG-11,EEG-12,EEG-13,EEG-14,EEG-Pz,EEG-15,EEG-16
0,1,-0.100,tongue,8,-1.681412,2.245496,-0.158350,1.163765,-1.523659,-0.575267,...,0.758116,3.441785,0.305517,1.137473,-1.275763,-2.898359,0.656704,-2.010063,-1.613804,-1.942455
1,1,-0.096,tongue,8,0.420417,0.587559,1.650510,0.970672,1.505904,0.891796,...,1.541586,-0.071620,0.258909,-1.448198,0.142472,-1.968405,-1.733655,-2.935578,-3.125256,-4.674610
2,1,-0.092,tongue,8,0.551365,1.499758,0.121302,2.859433,2.613414,4.636026,...,2.649097,-2.137938,-1.612096,-1.610218,-0.410173,-0.274957,-4.776535,-5.099551,-2.798995,-5.862021
3,1,-0.088,tongue,8,3.054916,-1.807238,1.843603,2.286812,5.995872,6.651295,...,6.031554,-5.249621,-2.672998,-3.452370,0.189081,1.593829,-6.081577,-5.476860,-2.932163,-6.874095
4,1,-0.084,tongue,8,2.506710,-2.453101,0.221178,0.127278,4.519931,6.249573,...,7.827097,-5.309546,-2.488783,-3.707608,1.447515,4.268278,-4.383690,-4.218426,-1.331932,-5.322692
5,1,-0.080,tongue,8,1.150619,-2.051379,-0.402491,0.529000,3.554466,4.551686,...,8.082335,-3.882433,-1.354639,-2.719949,0.823847,3.986407,-3.835483,-3.523735,-2.443882,-7.118236
6,1,-0.076,tongue,8,-1.537147,-2.151255,-2.943773,-0.498610,1.062012,4.061185,...,5.736366,-1.345590,-1.503343,-1.599121,2.725924,4.521297,-0.370906,-0.205642,0.092961,-2.921236
7,1,-0.072,tongue,8,-2.979796,-4.765779,-4.825875,-3.308447,-1.503684,1.934942,...,8.395279,1.362151,0.667289,-0.063255,1.234447,4.934116,2.043867,1.379053,1.384687,-0.945917
8,1,-0.068,tongue,8,-4.742048,-2.963577,-6.099845,-3.312886,-5.072576,1.491050,...,4.484590,4.580368,-0.460197,0.518244,1.718289,4.734365,4.041381,2.790630,2.063842,1.979332
9,1,-0.064,tongue,8,-5.061650,-1.964820,-5.052260,-5.390301,-6.271085,-1.855896,...,0.747019,5.383813,1.270982,2.102939,0.080327,2.461637,5.626076,3.887043,3.013771,4.589417


# **Preprocessing**

In [106]:
df['label'].unique()

array(['tongue', 'foot', 'right', 'left'], dtype=object)

In [107]:
df[eeg_columns] = df[eeg_columns].apply(pd.to_numeric, errors='coerce')
df = df.dropna()

# Check class distribution
print("Class distribution before windowing:")
print(df['label'].value_counts())

Class distribution before windowing:
label
right     115776
left      115776
tongue    101304
foot      101304
Name: count, dtype: int64


In [108]:
# Group by epochs
grouped = df.groupby('epoch')
X, y = [], []
target_length = 201

for epoch, group in grouped:
    data = group[eeg_columns].values.T
    if data.shape[1] > target_length:
        data = data[:, :target_length]
    elif data.shape[1] < target_length:
        pad_width = target_length - data.shape[1]
        data = np.pad(data, ((0, 0), (0, pad_width)), mode='constant')
    X.append(data)
    y.append(group['label'].iloc[0])

X = np.array(X)
y = np.array(y)
print(f"X shape after epoch grouping: {X.shape}")
print(f"y shape: {y.shape}")

X shape after epoch grouping: (611, 22, 201)
y shape: (611,)


In [109]:
# Channel-wise normalization
n_samples, n_channels, n_time_points = X.shape
X_normalized = np.zeros_like(X)
for i in range(n_samples):
    for ch in range(n_channels):
        scaler = StandardScaler()
        X_normalized[i, ch, :] = scaler.fit_transform(X[i, ch, :].reshape(-1, 1)).flatten()

In [110]:
# Frequency-domain features
X_freq = np.zeros((n_samples, n_channels, n_time_points // 2 + 1))
for i in range(n_samples):
    for ch in range(n_channels):
        fft_result = fft(X_normalized[i, ch, :])
        X_freq[i, ch, :] = np.abs(fft_result[:n_time_points // 2 + 1])

In [111]:
# Sliding window
window_size = 50
stride = 10
X_windows, y_windows = [], []

for i in range(n_samples):
    data = X_normalized[i]
    freq_data = X_freq[i]
    num_windows = (n_time_points - window_size) // stride + 1

    for w in range(num_windows):
        start = w * stride
        end = start + window_size
        window = data[:, start:end]

        freq_start = (start * (n_time_points // 2 + 1)) // n_time_points
        freq_end = (end * (n_time_points // 2 + 1)) // n_time_points
        freq_window = freq_data[:, freq_start:freq_end]

        if freq_window.shape[1] < window_size:
            freq_window = np.pad(freq_window, ((0, 0), (0, window_size - freq_window.shape[1])), mode='constant')
        elif freq_window.shape[1] > window_size:
            freq_window = freq_window[:, :window_size]

        combined = np.stack([window, freq_window], axis=-1)
        X_windows.append(combined)
        y_windows.append(y[i])

X_windows = np.array(X_windows)
y_windows = np.array(y_windows)
print(f"X_windows shape (before reshape): {X_windows.shape}")

X_windows shape (before reshape): (9776, 22, 50, 2)


In [112]:
# Encode labels
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y_windows)
encoder = LabelBinarizer()
y_categorical = encoder.fit_transform(y_encoded)
print(f"Encoded y shape (categorical): {y_categorical.shape}")

Encoded y shape (categorical): (9776, 4)


# Reshape: (samples, time_steps, channels, features) -> (samples, time_steps, channels * features)

In [113]:
X_windows = X_windows.transpose(0, 1, 3, 2)
print(f"X_windows shape: {X_windows.shape}")

X_windows shape: (9776, 22, 2, 50)


In [114]:
X_windows = X_windows.reshape(X_windows.shape[0], X_windows.shape[1]*X_windows.shape[2], X_windows.shape[3])
print(f"X_windows shape: {X_windows.shape}")

X_windows shape: (9776, 44, 50)


In [115]:
X_windows = np.array(X_windows)

In [116]:
#X_windows = torch.FloatTensor(X_windows)
y_categorical = torch.FloatTensor(y_categorical)
#y_encoded = y_integer = np.argmax(y_categorical, axis=1)

In [117]:
#np.save("F:/Project-Motor Imagery Classification/BCICIV_2a_gdf/Tensor/XLoso_8.npy", X_windows)
np.save(f"F:/Project-Motor Imagery Classification/BCICIV_2a_gdf/Tensor/yLoso_{sub}.npy", y_categorical)